In [ ]:
# -*- coding: utf-8 -*-
#
#  Copyright 2026 United Kingdom Research and Innovation
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.
#
#   Authored by:    Franck Vidal (UKRI-STFC)

![gVXR](https://github.com/TomographicImaging/gVXR-Tutorials/blob/main/img/Logo-transparent-small.png?raw=1)

# Digital twin of LabCT devices

blablabla

<div class="alert alert-block alert-warning">
    <b>Note:</b> Make sure the Python packages are already installed. See <a href="../README.md">README.md</a> in the root directory of the repository. If you are running this notebook from Google Colab, please run the cell below to install the required packages.
</div>

## Aim and objectives of this session

Import gVXR's main package and the functions from the main package needed for utilising the digital twin framework.

In [ ]:
from gvxrPython3 import gvxr
from gvxrPython3.twins.utils import getDigitalTwinList, createDigitalTwin
# from gvxrPython3.twins.utils import getAvailableFilters, getFilterMaterial, getFilterThickness

List all the twins available in the framework.

In [ ]:
available_twins = getDigitalTwinList()

# Process every twin
for i, twin in enumerate(available_twins):
    print("%i. %s" % (i+1, twin["name"]))
    print("\t- Available at:", twin["facility"])

    # Process every beam
    print("\t- Available beams:")
    for j, beam_label in enumerate(twin["beams"]):
        beam = twin["beams"][beam_label]
        beam_type = beam["beam_type"]
        print("\t\t%i. \"%s\"" % (j+1, beam_label))
        print("\t\t\t- Type:", beam_type)

        if beam_type == "monochromatic":
            print("\t\t\t- Energy range [in keV]:", beam["keV"])
        elif beam_type == "tube":
            print("\t\t\t- Voltage range [in kV]:", beam["kV"])

    detectors = twin["detector"]
    print("\t- Pixel pitch [in um]:", detectors["pixel_pitch"])

    resolutions = detectors["resolutions"]
    if len(resolutions) == 1:
        print("\t- Available resolutions: %ix%i" % (resolutions[0][0], resolutions[0][1]))
    else:
        print("\t- Available resolutions:")
        for k, resolution in enumerate(detectors["resolutions"]):
            print("\t\t%i. %ix%i" % (k+1, resolution[0], resolution[1]))
    if detectors["scintillator"] != None and detectors["scintillator"] != "":
        print("\t- Scintillator: %i um of %s" %
              (detectors["scintillator"]["thickness"], detectors["scintillator"]["material"])
        )
    else:
        print("\t- Scintillator: unknown")


We will make use of the DTHE twin. This is the most developed twin that we got.

In [ ]:
twin_name = "DTHE"
twin = createDigitalTwin(name=twin_name)


Print the characteristics of this twin

In [ ]:
print(twin)

Create an OpenGL context and initialise the simulation engine.

In [ ]:
gvxr.createNewContext()

## Tube voltage

There is only one beam. There is no need to select it, but we can change the voltage.

In [ ]:
print("Number of beams:", len(twin.specification.beams.items()))

beam_name = list(twin.specification.beams)[0]
print("Its name is \"%s\"" % beam_name)

print("Valid voltage range [in kV]:", twin.specification.beams[beam_name].kV)
print("Default voltage [in kV]:", twin.beam.kV)

Change the voltage to 200 kV

In [ ]:
twin.beam.kV = 200

## Tube current

Check what current values are allowed and what the default value is

In [ ]:
twin_max_uA = max(twin.specification.beams[beam_name].uA)
twin_min_uA = min(twin.specification.beams[beam_name].uA)
current_uA_value = twin.beam.uA

print("%f <= %f <= %f" % (twin_min_uA, current_uA_value, twin_max_uA))

Change the current to 250 uA

In [ ]:
twin.beam.uA = 250

## Detector exposure

Check what exposure times are allowed, and what the default value is

In [ ]:
exposures = sorted(twin.specification.detector.exposures)
print("Valid exposures:", exposures)
print("Current exposure:", twin.detector.exposure)

Let's use 0.111

In [ ]:
twin.detector.exposure = 0.111

# Detector gain

Check what gains are allowed, and what the default value is


In [ ]:
print("Valid gain values:", twin.specification.detector.gain.adus)
print("Default gain: %i. It is an index to the array above." % twin.detector.gain)
print("If you want to use 114, use `twin.detector.gain = 2`.")

We will use 229. Change the value of `twin.detector.gain` accordingly.

In [ ]:
twin.detector.gain = 3

Apply the current setting

In [ ]:
twin.apply()

In [ ]:
gvxr.makeFoamInCuboid("foam_volume_id",
    "", # The parameter "aBoundingVolumeName" is not used.
    50, 50, 50,
    5,
    50,
    0.2,
    1.0,
    "mm",
    1)

In [ ]:
# Use implicit surface to create foam mesh
gvxr.emptyMesh("sample")
gvxr.createMeshFromImplicitSurface("foam_volume_id", "matrix", 0.5, "sample", 1)
gvxr.createMeshFromImplicitSurface("foam_volume_id", "porosity", -10.0, 0.5, "sample", 1)

In [ ]:
gvxr.setCompound("porosity", "SiC")
gvxr.setDensity("porosity", 3.16, "g/cm3")

gvxr.setElement("matrix", "Al")
#
gvxr.addPolygonMeshAsInnerSurface("porosity")
gvxr.addPolygonMeshAsInnerSurface("matrix")

In [ ]:
optimal_sod = gvxr.getBestSOD("sample", "cm")
gvxr.applySOD("sample", max(5, optimal_sod), "cm")

In [ ]:
# gvxr.setEnergyBinSize(2, "keV")

In [ ]:
import numpy as np
# xray_image_float = np.asarray(gvxr.computeXRayImage(), dtype=np.single) / gvxr.getTotalEnergyWithDetectorResponse(True)

In [ ]:
# xray_image_ushort = np.asarray(gvxr.computeXRayImageWithGain(), dtype=np.uint16)

In [ ]:
# from matplotlib import pyplot as plt
#
# plt.figure()
# plt.imshow(xray_image_float)
# plt.colorbar()
# plt.show()
#
# plt.figure()
# plt.imshow(xray_image_ushort)
# plt.colorbar()
# plt.show()

Saturate the detector?

In [ ]:
# np.max(xray_image_ushort)

Print the list of filters supported by the twin.


In [ ]:
# for spectrum in twin.get_beams():
#
#     filter_materials = getFilterMaterial(twin, spectrum)
#     if len(filter_materials) == 0:
#         print("\tSupported filters: None")
#     else:
#         print("\tSupported filters")
#         for material in filter_materials:
#             for thickness in getFilterThickness(twin, spectrum, material):
#                 if material != "None":
#                     print("\t\t%s mm of %s" % (thickness, material))

In [ ]:
twin.detector.gain = 3
twin.detector.exposure = 0.067
twin.beam.uA = 500
twin.beam.kV = 180

twin.apply()

# gvxr.clearFiltration()
gvxr.addFilter("Ag", 0.5, "mm")

# print(gvxr.getInherentFiltrationMaterial())
# print(gvxr.getInherentFiltrationThickness("mm"))
print(gvxr.getFiltrationMaterial())
print(gvxr.getFiltrationThickness("mm"))
print(gvxr.getTotalEnergyWithDetectorResponseWithGain())
print(gvxr.getTotalEnergyWithDetectorResponse())
print(gvxr.getTotalEnergyWithDetectorResponseWithGain() / 4000)



(13, 6)
(0.5, 4.0)
(13,)
(0.5,)
13469
29.76055908203125

In [ ]:
bins = gvxr.getEnergyBins("keV")
counts = gvxr.getPhotonCountsPerPixelAtSDD()

plt.figure()
plt.plot(bins, counts)

In [ ]:
twin.detector.gain = 3
twin.detector.exposure = 0.067
twin.beam.uA = 500
twin.beam.kV = 180

twin.apply()

gvxr.clearFiltration()
gvxr.addFilter("Cu", 0.5, "mm")

# print(gvxr.getInherentFiltrationMaterial())
# print(gvxr.getInherentFiltrationThickness("mm"))
print(gvxr.getFiltrationMaterial())
print(gvxr.getFiltrationThickness("mm"))
print(gvxr.getTotalEnergyWithDetectorResponseWithGain())
print(gvxr.getTotalEnergyWithDetectorResponseWithGain() / 11000)

In [ ]:
bins = gvxr.getEnergyBins("keV")
counts = gvxr.getPhotonCountsPerPixelAtSDD()

plt.figure()
plt.plot(bins, counts)

In [ ]:
twin.detector.gain = 3
twin.detector.exposure = 0.067
twin.beam.uA = 500
twin.beam.kV = 260

twin.apply()

gvxr.clearFiltration()
print(gvxr.getNumberOfPhotonsPerCm2At1m())



In [ ]:
twin.detector.gain = 3
twin.detector.exposure = 0.167
twin.beam.uA = 200
twin.beam.kV = 160

gvxr.clearFiltration()

twin.apply()
gvxr.setSourcePosition(0, 0, -807.249 / 2, "mm")
gvxr.setDetectorPosition(0, 0, 807.249 / 2, "mm")
print(gvxr.getSDD("mm"))
print(gvxr.getTotalEnergyWithDetectorResponseWithGain())
print(gvxr.getTotalEnergyWithDetectorResponseWithGain() / 44061)

44061

In [ ]:
bins = gvxr.getEnergyBins("keV")
counts = gvxr.getPhotonCountsPerPixelAtSDD()

plt.figure()
plt.plot(bins, counts)


In [ ]:
for bin, count in zip(bins, counts):
    print(bin, count)